In [1]:
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import transforms, datasets
from torch.utils.data import Dataset, DataLoader
import pytorch_lightning as pl
from pytorch_lightning import Trainer
from torchmetrics.classification import MulticlassAccuracy
import torchvision.models as models

from PIL import Image

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using Device: {DEVICE}")

Using Device: cuda


In [2]:
# JUST OUR USUAL DATASET CLASS
class BrainTumorDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        """Initializes the dataset object. Takes path and transform as inputs"""        
        super().__init__()        
        self.root_dir = root_dir
        self.transform = transform     
        self.classes = sorted(os.listdir(root_dir))        # Manually map folder names to integers
        self.class_to_idx = {cls_name: i for i, cls_name in enumerate(self.classes)}
        
        # 3. Build the file list
        self.images = []
        for cls_name in self.classes:
            cls_path = os.path.join(root_dir, cls_name)
            for img_name in os.listdir(cls_path):
                self.images.append((os.path.join(cls_path, img_name), self.class_to_idx[cls_name]))

    def __len__(self):
        return len(self.images)         # Returns the number of images in the dataset

    def __getitem__(self, idx):
        """Grabs a sepcific image, loads it and applies transforms on it"""
        img_path, label = self.images[idx]
        image = Image.open(img_path).convert("RGB")       
        if self.transform:
            image = self.transform(image)
            
        return image, label


In [3]:
### LOADING THE DATA USING DATALOADER
data_path = '/kaggle/input/datasets/masoudnickparvar/brain-tumor-mri-dataset' ### Dataset folder path
## Adding transforms
mri_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
])

# Instantiate Datasets using the path of the data
train_dataset = BrainTumorDataset(os.path.join(data_path, 'Training'), transform=mri_transforms)
test_dataset = BrainTumorDataset(os.path.join(data_path, 'Testing'), transform=mri_transforms)

# Finally the DataLoader to create and load batches for model training. 
# Note how with a smaller dataset, loading a batch size of 32 is also feasible!
train_loader = DataLoader(
    train_dataset, 
    batch_size=32, 
    shuffle=True, 
    num_workers=os.cpu_count(), 
    pin_memory=True
)
val_loader = DataLoader(
    test_dataset, 
    batch_size=32, 
    shuffle=False, 
    num_workers=os.cpu_count(), 
    pin_memory=True
)
print(f"Loaded {len(train_dataset)} training images across classes: {train_dataset.classes}")

Loaded 5600 training images across classes: ['glioma', 'meningioma', 'notumor', 'pituitary']


In [4]:
class BrainTumorClassifier(pl.LightningModule):
    """Define the model architecture, the forward pass, the train/val step and optimizer"""
    def __init__(self, num_classes=4):
        super().__init__()        
        # Load a pretrained backbone, extract the number of input features and replace the final layer to fit 
        # the classifier head
        backbone = models.resnet50(weights="DEFAULT")        
        # Extract the number of input features from the last layer (fc)
        num_filters = backbone.fc.in_features
        self.feature_extractor = nn.Sequential(*list(backbone.children())[:-1])
        ## Defining the final layer as classification head
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(num_filters, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, num_classes)
        )
        self.train_acc = MulticlassAccuracy(num_classes=num_classes)
        self.val_acc = MulticlassAccuracy(num_classes=num_classes)

    def forward(self, x):
        x = self.feature_extractor(x) ## Using backbone as feature extractor and classification head on the output
        return self.classifier(x)
        
    def training_step(self, batch, batch_idx):
        x, y = batch        
        logits = self(x)
        loss = F.cross_entropy(logits, y) ## Calculate loss
        acc = self.train_acc(logits, y) ## Calculate accuracy   
        # For logs and display
        self.log("train_loss", loss, prog_bar=True, on_step=True, on_epoch=True)
        self.log("train_acc", acc, prog_bar=True, on_step=True, on_epoch=True)      
        return loss    

    def validation_step(self, batch, batch_idx):
        x, y = batch
        logits = self(x)
        loss = F.cross_entropy(logits, y)
        acc = self.val_acc(logits, y)
        
        # Adding prog_bar=True makes these appear in the progress bar
        self.log("val_loss", loss, prog_bar=True)
        self.log("val_acc", acc, prog_bar=True)
    
    def configure_optimizers(self):
        return torch.optim.Adam(self.parameters(), lr=1e-3)

In [5]:

model = BrainTumorClassifier(num_classes=4) ### Defining the model

trainer = Trainer(max_epochs=40,precision='16-mixed',accelerator="auto")

# Check the precision policy
print(f"Active Precision Plugin: {trainer.precision_plugin.__class__.__name__}")
print(f"Precision Setting: {trainer.precision}")

# Start training
trainer.fit(model, train_loader, val_loader)


Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 190MB/s] 
Trainer will use only 1 of 2 GPUs because it is running inside an interactive / notebook environment. You may try to set `Trainer(devices=2)` but please note that multi-GPU inside interactive / notebook environments is considered experimental and unstable. Your mileage may vary.
Using 16bit Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.


Active Precision Plugin: MixedPrecision
Precision Setting: 16-mixed


2026-05-18 05:35:37.978075: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1779082538.166026      57 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1779082538.218678      57 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1779082538.686575      57 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1779082538.686641      57 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1779082538.686648      57 computation_placer.cc:177] computation placer alr

┏━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name              ┃ Type               ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ feature_extractor │ Sequential         │ 23.5 M │ train │     0 │
│ 1 │ classifier        │ Sequential         │  525 K │ train │     0 │
│ 2 │ train_acc         │ MulticlassAccuracy │      0 │ train │     0 │
│ 3 │ val_acc           │ MulticlassAccuracy │      0 │ train │     0 │
└───┴───────────────────┴────────────────────┴────────┴───────┴───────┘

Trainable params: 24.0 M                                                                                           
Non-trainable params: 0                                                                                            
Total params: 24.0 M                                                                                               
Total estimated model params size (MB): 96                                                                         
Modules in train mode: 158                                                                                         
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

/usr/local/lib/python3.12/dist-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)`
is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.

`Trainer.fit` stopped: `max_epochs=40` reached.
